# Smart Hospitality Management System
## Module 2: Hotel Review Sentiment Analysis

End-to-end pipeline: cleaning -> EDA -> preprocessing -> TF-IDF baseline -> BiLSTM deep learning model.

## 1. Data Cleaning & Sentiment Labeling

In [ ]:
"""
Module 2: Hotel Review Sentiment Analysis
Step 1: Data Cleaning & Sentiment Labeling
"""
import pandas as pd
import numpy as np
import re

df = pd.read_csv('data/tripadvisor_hotel_reviews.csv')
print(f"Original shape: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()}")

# ------------------------------------------------------------------
# 1. Derive sentiment label from star rating
# ------------------------------------------------------------------
def rating_to_sentiment(r):
    if r <= 2:
        return 'negative'
    elif r == 3:
        return 'neutral'
    else:
        return 'positive'

df['sentiment'] = df['Rating'].apply(rating_to_sentiment)
print(f"\nSentiment distribution:\n{df['sentiment'].value_counts()}")
print(f"\nSentiment distribution (%):\n{df['sentiment'].value_counts(normalize=True).round(3)}")

# ------------------------------------------------------------------
# 2. Basic dedup / empty-text checks
# ------------------------------------------------------------------
before = len(df)
df = df.drop_duplicates(subset=['Review'])
print(f"\nDropped {before - len(df)} duplicate reviews")

df['review_len_words'] = df['Review'].apply(lambda x: len(str(x).split()))
before = len(df)
df = df[df['review_len_words'] >= 3]   # drop near-empty reviews
print(f"Dropped {before - len(df)} reviews with <3 words")

# ------------------------------------------------------------------
# 3. Text cleaning function
# ------------------------------------------------------------------
# Small custom stopword list (no internet access for nltk downloads in this env;
# in your own environment you can swap this for nltk.corpus.stopwords)
STOPWORDS = set("""a an the and or but if while of at by for with about against
between into through during before after above below to from up down in out on off
over under again further then once here there all any both each few more most other
some such no nor not only own same so than too very s t can will just don should now
i me my myself we our ours ourselves you your yours yourself yourselves he him his
himself she her hers herself it its itself they them their theirs themselves what
which who whom this that these those am is are was were be been being have has had
having do does did doing would could should ought i'm you're he's she's it's we're
they're""".split())

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)          # remove punctuation/numbers
    text = re.sub(r"\s+", " ", text).strip()        # collapse whitespace
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return " ".join(tokens)

df['clean_review'] = df['Review'].apply(clean_text)

# drop rows that became empty after cleaning
before = len(df)
df = df[df['clean_review'].str.len() > 0]
print(f"Dropped {before - len(df)} reviews that were empty after cleaning")

print(f"\nFinal cleaned shape: {df.shape}")
print("\nSample cleaned review:")
print("Original:", df['Review'].iloc[0][:150])
print("Cleaned: ", df['clean_review'].iloc[0][:150])

df.to_csv('data/reviews_cleaned.csv', index=False)
print("\nSaved -> data/reviews_cleaned.csv")


## 2. Exploratory Data Analysis

In [ ]:
"""
Module 2: Hotel Review Sentiment Analysis
Step 2: Exploratory Data Analysis
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

df = pd.read_csv('data/reviews_cleaned.csv')

# ------------------------------------------------------------------
# 1. Class distribution
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
order = ['negative', 'neutral', 'positive']
colors = ['#C44E52', '#DD8452', '#55A868']
df['sentiment'].value_counts().reindex(order).plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Sentiment Class Distribution')
axes[0].set_xticklabels(order, rotation=0)

df['Rating'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='#4C72B0')
axes[1].set_title('Star Rating Distribution')
axes[1].set_xticklabels(range(1, 6), rotation=0)
plt.tight_layout()
plt.savefig('plots/01_class_distribution.png')
plt.close()

# ------------------------------------------------------------------
# 2. Review length by sentiment
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=df, x='sentiment', y='review_len_words', order=order, palette=colors, ax=ax)
ax.set_ylim(0, 400)
ax.set_title('Review Length (words) by Sentiment')
plt.tight_layout()
plt.savefig('plots/02_review_length.png')
plt.close()

# ------------------------------------------------------------------
# 3. Most frequent words per sentiment class (bar charts in place of wordcloud
#    - no internet access in this sandbox to install the wordcloud package)
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, sent, color in zip(axes, order, colors):
    text = " ".join(df[df.sentiment == sent]['clean_review'])
    counts = Counter(text.split())
    top15 = counts.most_common(15)
    words, freqs = zip(*top15)
    ax.barh(words[::-1], freqs[::-1], color=color)
    ax.set_title(f'Top Words - {sent.capitalize()}')
plt.tight_layout()
plt.savefig('plots/03_top_words_by_sentiment.png')
plt.close()

# ------------------------------------------------------------------
# 4. Summary stats
# ------------------------------------------------------------------
print("Average review length by sentiment:")
print(df.groupby('sentiment')['review_len_words'].mean().round(1))
print("\nVocabulary size (unique words) in cleaned corpus:")
all_words = " ".join(df['clean_review']).split()
print(f"  Total tokens: {len(all_words)}, Unique tokens: {len(set(all_words))}")

print("\nPlots saved to plots/ directory")


## 3. Preprocessing (Label Encoding, Split, TF-IDF)

In [ ]:
"""
Module 2: Hotel Review Sentiment Analysis
Step 3: Preprocessing (label encoding, train/test split, TF-IDF)
"""
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import joblib

df = pd.read_csv('data/reviews_cleaned.csv')

# ------------------------------------------------------------------
# 1. Encode sentiment labels
# ------------------------------------------------------------------
le = LabelEncoder()
df['label'] = le.fit_transform(df['sentiment'])   # negative=0, neutral=1, positive=2 (alphabetical)
print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# ------------------------------------------------------------------
# 2. Train/test split (stratified — important given class imbalance)
# ------------------------------------------------------------------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_review'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)
print(f"\nTrain size: {len(X_train_text)}, Test size: {len(X_test_text)}")

# ------------------------------------------------------------------
# 3. TF-IDF vectorization (for baseline Logistic Regression)
# ------------------------------------------------------------------
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=3)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)
print(f"TF-IDF matrix shape: train {X_train_tfidf.shape}, test {X_test_tfidf.shape}")

# ------------------------------------------------------------------
# 4. Save everything
# ------------------------------------------------------------------
X_train_text.to_csv('data/X_train_text.csv', index=False)
X_test_text.to_csv('data/X_test_text.csv', index=False)
y_train.to_csv('data/y_train.csv', index=False)
y_test.to_csv('data/y_test.csv', index=False)

joblib.dump(X_train_tfidf, 'data/X_train_tfidf.pkl')
joblib.dump(X_test_tfidf, 'data/X_test_tfidf.pkl')
joblib.dump(tfidf, 'models/tfidf_vectorizer.pkl')
joblib.dump(le, 'models/label_encoder.pkl')

print("\nSaved preprocessed data, TF-IDF vectorizer, and label encoder")


## 4. Baseline Model (TF-IDF + Logistic Regression)

In [ ]:
"""
Module 2: Hotel Review Sentiment Analysis
Step 4: Baseline Model (TF-IDF + Logistic Regression)
"""
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

X_train_tfidf = joblib.load('data/X_train_tfidf.pkl')
X_test_tfidf = joblib.load('data/X_test_tfidf.pkl')
y_train = pd.read_csv('data/y_train.csv').values.ravel()
y_test = pd.read_csv('data/y_test.csv').values.ravel()
le = joblib.load('models/label_encoder.pkl')

# ------------------------------------------------------------------
# 1. Train Logistic Regression (multi-class, class-weighted for imbalance)
# ------------------------------------------------------------------
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_tfidf, y_train)

y_pred = lr.predict(X_test_tfidf)

# ------------------------------------------------------------------
# 2. Evaluate
# ------------------------------------------------------------------
print("--- TF-IDF + Logistic Regression: Test Set Performance ---")
print(f"Accuracy:           {accuracy_score(y_test, y_pred):.4f}")
print(f"Macro Precision:    {precision_score(y_test, y_pred, average='macro'):.4f}")
print(f"Macro Recall:       {recall_score(y_test, y_pred, average='macro'):.4f}")
print(f"Macro F1:            {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"Weighted F1:         {f1_score(y_test, y_pred, average='weighted'):.4f}")

print("\nPer-class report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# ------------------------------------------------------------------
# 3. Confusion matrix
# ------------------------------------------------------------------
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=le.classes_, yticklabels=le.classes_)
ax.set_title('TF-IDF + Logistic Regression - Confusion Matrix')
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('plots/04_baseline_confusion_matrix.png')
plt.close()

# ------------------------------------------------------------------
# 4. Save results & model
# ------------------------------------------------------------------
results = {
    'accuracy': accuracy_score(y_test, y_pred),
    'macro_precision': precision_score(y_test, y_pred, average='macro'),
    'macro_recall': recall_score(y_test, y_pred, average='macro'),
    'macro_f1': f1_score(y_test, y_pred, average='macro'),
    'weighted_f1': f1_score(y_test, y_pred, average='weighted'),
}
pd.Series(results).to_csv('models/baseline_results.csv')
joblib.dump(lr, 'models/logistic_regression_sentiment.pkl')
print("\nSaved model and results")


## 5. Deep Learning Model (BiLSTM — requires TensorFlow, run in Colab)

In [ ]:
"""
Module 2: Hotel Review Sentiment Analysis
Step 5: Deep Learning Model (BiLSTM with Keras/TensorFlow)

>>> RUN THIS IN GOOGLE COLAB OR A LOCAL ENVIRONMENT WITH TENSORFLOW INSTALLED <<<
(pip install tensorflow)

Assumes data/reviews_cleaned.csv and data/X_train_text.csv / X_test_text.csv /
y_train.csv / y_test.csv from the earlier preprocessing step are available.
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
from sklearn.utils.class_weight import compute_class_weight
import joblib

tf.random.set_seed(42)
np.random.seed(42)

# ------------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------------
X_train_text = pd.read_csv('data/X_train_text.csv').iloc[:, 0].astype(str)
X_test_text = pd.read_csv('data/X_test_text.csv').iloc[:, 0].astype(str)
y_train = pd.read_csv('data/y_train.csv').values.ravel()
y_test = pd.read_csv('data/y_test.csv').values.ravel()
le = joblib.load('models/label_encoder.pkl')
num_classes = len(le.classes_)

# ------------------------------------------------------------------
# 2. Tokenization & padding
# ------------------------------------------------------------------
VOCAB_SIZE = 15000
MAX_LEN = 150   # covers most reviews; longer ones get truncated

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

print(f"X_train_pad shape: {X_train_pad.shape}, X_test_pad shape: {X_test_pad.shape}")

# handle class imbalance
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print(f"Class weights: {class_weight_dict}")

# ------------------------------------------------------------------
# 3. Build the BiLSTM
# ------------------------------------------------------------------
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=128, input_length=MAX_LEN),
    SpatialDropout1D(0.3),

    Bidirectional(LSTM(64, return_sequences=True)),
    Bidirectional(LSTM(32)),

    Dense(64, activation='relu'),
    Dropout(0.4),

    Dense(num_classes, activation='softmax')   # 3-class: negative/neutral/positive
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ------------------------------------------------------------------
# 4. Train with early stopping
# ------------------------------------------------------------------
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

history = model.fit(
    X_train_pad, y_train,
    validation_split=0.15,
    epochs=30,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

# ------------------------------------------------------------------
# 5. Evaluate on test set
# ------------------------------------------------------------------
y_pred_proba = model.predict(X_test_pad)
y_pred = np.argmax(y_pred_proba, axis=1)

print("\n--- BiLSTM Test Set Performance ---")
print(f"Accuracy:        {accuracy_score(y_test, y_pred):.4f}")
print(f"Macro Precision: {precision_score(y_test, y_pred, average='macro'):.4f}")
print(f"Macro Recall:    {recall_score(y_test, y_pred, average='macro'):.4f}")
print(f"Macro F1:        {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"Weighted F1:     {f1_score(y_test, y_pred, average='weighted'):.4f}")
print("\nPer-class report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# ------------------------------------------------------------------
# 6. Training curves
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train Acc')
axes[1].plot(history.history['val_accuracy'], label='Val Acc')
axes[1].set_title('Accuracy'); axes[1].legend()
plt.tight_layout()
plt.savefig('plots/05_bilstm_training_curves.png')
plt.close()

# ------------------------------------------------------------------
# 7. Confusion matrix
# ------------------------------------------------------------------
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=ax,
            xticklabels=le.classes_, yticklabels=le.classes_)
ax.set_title('BiLSTM - Confusion Matrix')
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('plots/06_bilstm_confusion_matrix.png')
plt.close()

# ------------------------------------------------------------------
# 8. Save model & tokenizer
# ------------------------------------------------------------------
model.save('models/bilstm_sentiment_model.keras')
joblib.dump(tokenizer, 'models/tokenizer.pkl')
print("\nModel and tokenizer saved")


## 6. Improved Deep Learning Model — BiLSTM + Pretrained GloVe Embeddings (requires TensorFlow + gensim, run in Colab)

In [ ]:
"""
Module 2: Hotel Review Sentiment Analysis
Step 5b: Improved Deep Learning Model — BiLSTM with Pretrained GloVe Embeddings

>>> RUN THIS IN GOOGLE COLAB (needs internet access + TensorFlow) <<<

Why this should help: the plain BiLSTM (05_bilstm_model.py) trains its word
embeddings from scratch on only ~16K reviews, which usually isn't enough data
to learn good general word representations. GloVe embeddings are pretrained
on 6 billion tokens of real text, so words the model rarely saw in training
(e.g. "immaculate", "filthy", "unresponsive") already carry meaningful vector
representations before training even starts.

Setup (run once in Colab):
    pip install gensim
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
from sklearn.utils.class_weight import compute_class_weight
import joblib
import gensim.downloader as gensim_api

tf.random.set_seed(42)
np.random.seed(42)

# ------------------------------------------------------------------
# 1. Load data (reuse the same split as the baseline BiLSTM for a fair comparison)
# ------------------------------------------------------------------
X_train_text = pd.read_csv('data/X_train_text.csv').iloc[:, 0].astype(str)
X_test_text = pd.read_csv('data/X_test_text.csv').iloc[:, 0].astype(str)
y_train = pd.read_csv('data/y_train.csv').values.ravel()
y_test = pd.read_csv('data/y_test.csv').values.ravel()
le = joblib.load('models/label_encoder.pkl')
num_classes = len(le.classes_)

# ------------------------------------------------------------------
# 2. Tokenization & padding (same settings as before)
# ------------------------------------------------------------------
VOCAB_SIZE = 15000
MAX_LEN = 150
EMBEDDING_DIM = 100   # matches glove-wiki-gigaword-100

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

# ------------------------------------------------------------------
# 3. Load pretrained GloVe vectors & build embedding matrix
# ------------------------------------------------------------------
print("Downloading GloVe vectors (glove-wiki-gigaword-100)... this may take a minute.")
glove_vectors = gensim_api.load('glove-wiki-gigaword-100')

word_index = tokenizer.word_index
vocab_size = min(VOCAB_SIZE, len(word_index) + 1)
embedding_matrix = np.zeros((vocab_size, EMBEDDING_DIM))

hits, misses = 0, 0
for word, i in word_index.items():
    if i >= vocab_size:
        continue
    if word in glove_vectors:
        embedding_matrix[i] = glove_vectors[word]
        hits += 1
    else:
        misses += 1   # stays as zero vector -> model still learns it during training

print(f"Converted {hits} words ({hits / (hits + misses):.1%}); missed {misses} words")

# ------------------------------------------------------------------
# 4. Build the BiLSTM with GloVe-initialized embedding layer
# ------------------------------------------------------------------
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=EMBEDDING_DIM, input_length=MAX_LEN,
              weights=[embedding_matrix], trainable=True),   # fine-tune embeddings during training
    SpatialDropout1D(0.3),

    Bidirectional(LSTM(64, return_sequences=True)),
    Bidirectional(LSTM(32)),

    Dense(64, activation='relu'),
    Dropout(0.4),

    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

# ------------------------------------------------------------------
# 5. Train
# ------------------------------------------------------------------
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

history = model.fit(
    X_train_pad, y_train,
    validation_split=0.15,
    epochs=30,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

# ------------------------------------------------------------------
# 6. Evaluate
# ------------------------------------------------------------------
y_pred_proba = model.predict(X_test_pad)
y_pred = np.argmax(y_pred_proba, axis=1)

print("\n--- BiLSTM + GloVe Test Set Performance ---")
print(f"Accuracy:        {accuracy_score(y_test, y_pred):.4f}")
print(f"Macro Precision: {precision_score(y_test, y_pred, average='macro'):.4f}")
print(f"Macro Recall:    {recall_score(y_test, y_pred, average='macro'):.4f}")
print(f"Macro F1:        {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"Weighted F1:     {f1_score(y_test, y_pred, average='weighted'):.4f}")
print("\nPer-class report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# ------------------------------------------------------------------
# 7. Training curves
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train Acc')
axes[1].plot(history.history['val_accuracy'], label='Val Acc')
axes[1].set_title('Accuracy'); axes[1].legend()
plt.tight_layout()
plt.savefig('plots/07_bilstm_glove_training_curves.png')
plt.close()

# ------------------------------------------------------------------
# 8. Confusion matrix
# ------------------------------------------------------------------
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=ax,
            xticklabels=le.classes_, yticklabels=le.classes_)
ax.set_title('BiLSTM + GloVe - Confusion Matrix')
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('plots/08_bilstm_glove_confusion_matrix.png')
plt.close()

# ------------------------------------------------------------------
# 9. Save model
# ------------------------------------------------------------------
model.save('models/bilstm_glove_sentiment_model.keras')
joblib.dump(tokenizer, 'models/tokenizer_glove.pkl')
print("\nModel saved")
